## Data Engineering Module 2 - Apache Spark Manipulations

### Loading CSV Files into the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading the CSV files into a Dataframe

In [0]:
# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8
SO49171,1,2021-01-01,Mariah Foster,mariah21@adventure-works.com,"Road-250 Black, 48",1,2181.5625,174.525
SO49172,1,2021-01-01,Brian Howard,brian23@adventure-works.com,"Road-250 Red, 44",1,2443.35,195.468
SO49173,1,2021-01-01,Linda Alvarez,linda19@adventure-works.com,"Mountain-200 Silver, 38",1,2071.4196,165.7136
SO49174,1,2021-01-01,Gina Hernandez,gina4@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4196,165.7136
SO49178,1,2021-01-01,Beth Ruiz,beth4@adventure-works.com,"Road-550-W Yellow, 44",1,1000.4375,80.035
SO49179,1,2021-01-01,Evan Ward,evan13@adventure-works.com,"Road-550-W Yellow, 38",1,1000.4375,80.035
SO49175,1,2021-01-01,Margaret Guo,margaret24@adventure-works.com,"Road-250 Red, 52",1,2443.35,195.468
SO49180,1,2021-01-01,Mitchell Yuan,mitchell6@adventure-works.com,"Road-650 Black, 58",1,782.99,62.6392
SO49176,1,2021-01-01,Shawn Sharma,shawn11@adventure-works.com,"Mountain-200 Silver, 46",1,2071.4196,165.7136
SO49177,1,2021-01-01,Barbara Chande,barbara44@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4196,165.7136


### Defining Schema for the Dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv', schema=orderSchema)
display(df.limit(100))
     

SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,Email,Item,Quantity,UnitPrice,Tax
SO49171,1,2021-01-01,Mariah Foster,mariah21@adventure-works.com,"Road-250 Black, 48",1,2181.5625,174.525
SO49172,1,2021-01-01,Brian Howard,brian23@adventure-works.com,"Road-250 Red, 44",1,2443.35,195.468
SO49173,1,2021-01-01,Linda Alvarez,linda19@adventure-works.com,"Mountain-200 Silver, 38",1,2071.4197,165.7136
SO49174,1,2021-01-01,Gina Hernandez,gina4@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4197,165.7136
SO49178,1,2021-01-01,Beth Ruiz,beth4@adventure-works.com,"Road-550-W Yellow, 44",1,1000.4375,80.035
SO49179,1,2021-01-01,Evan Ward,evan13@adventure-works.com,"Road-550-W Yellow, 38",1,1000.4375,80.035
SO49175,1,2021-01-01,Margaret Guo,margaret24@adventure-works.com,"Road-250 Red, 52",1,2443.35,195.468
SO49180,1,2021-01-01,Mitchell Yuan,mitchell6@adventure-works.com,"Road-650 Black, 58",1,782.99,62.6392
SO49176,1,2021-01-01,Shawn Sharma,shawn11@adventure-works.com,"Mountain-200 Silver, 46",1,2071.4197,165.7136
SO49177,1,2021-01-01,Barbara Chande,barbara44@adventure-works.com,"Mountain-200 Silver, 42",1,2071.4197,165.7136


### Cleaning the Data

In [0]:
from pyspark.sql.functions import col
df = df.dropDuplicates()
df = df.withColumn('Tax', col('UnitPrice') * 0.08)
df = df.withColumn('Tax', col('Tax').cast("float"))

### Creating a new Dataframe

In [0]:
customers_df = df.select("CustomerName", "Email", "Item", "Quantity")
customers_df = customers_df.withColumn("FirstName", split(customers_df["CustomerName"], " ").getItem(0))
customers_df = customers_df.withColumn("LastName", split(customers_df["CustomerName"], " ").getItem(1))

display(customers_df)

CustomerName,Email,Item,Quantity,FirstName,LastName
Beth Ruiz,beth4@adventure-works.com,"Road-550-W Yellow, 44",1,Beth,Ruiz
Derrick Jim nez,derrick5@adventure-works.com,"Road-250 Black, 48",1,Derrick,Jim
Tommy Raje,tommy10@adventure-works.com,"Road-550-W Yellow, 42",1,Tommy,Raje
Kevin Simmons,kevin18@adventure-works.com,"Mountain-200 Silver, 42",1,Kevin,Simmons
Alejandro Kumar,alejandro34@adventure-works.com,"Mountain-200 Black, 46",1,Alejandro,Kumar
Madison Foster,madison28@adventure-works.com,"Mountain-200 Black, 38",1,Madison,Foster
Chloe Turner,chloe2@adventure-works.com,"Mountain-200 Silver, 38",1,Chloe,Turner
Felicia Blanco,felicia15@adventure-works.com,"Mountain-200 Black, 38",1,Felicia,Blanco
Ian King,ian25@adventure-works.com,"Road-550-W Yellow, 38",1,Ian,King
Patrick Sands,patrick4@adventure-works.com,"Mountain-200 Silver, 38",1,Patrick,Sands


### Creating Distinct Customer Entries

In [0]:
print(customers_df.count())
print(customers_df.distinct().count())

32718
32200


### Creating a Product Sales Dataframe

In [0]:
productSales = df.select("Item", "Quantity").groupBy("Item").sum()
display(productSales)

Item,sum(Quantity)
"Road-550-W Yellow, 44",183
"Road-250 Black, 48",240
"Road-550-W Yellow, 42",180
"Mountain-200 Silver, 42",359
"Mountain-200 Black, 46",425
"Mountain-200 Black, 38",366
"Mountain-200 Silver, 38",370
"Road-550-W Yellow, 38",171
"Mountain-200 Black, 42",388
"Road-650 Black, 52",89


### Aggregating Yearly Sales

In [0]:
yearlySales = df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

Year,count
2019,1201
2020,2733
2021,28784
